In [1]:
# This notebook removes duplicate positions and makes fixed train / val / test splits for all models

import numpy as np
import pandas as pd
from pathlib import Path

out_dir = Path("splits")
out_dir.mkdir(exist_ok=True)

In [2]:
def fen_key(fens):
    return fens.str.split(" ").str[:4].str.join(" ")

def split(df, val_frac=0.1, test_frac=0.1, seed=42):
    df = df.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    n_val = int(val_frac * len(df))
    n_test = int(test_frac * len(df))
    test = df.iloc[:n_test]
    val = df.iloc[n_test:n_test + n_val]
    train = df.iloc[n_test + n_val:]
    return train, val, test

In [3]:
files = [
    "archive/FilteredEvals.csv",
    "archive/FilteredEvals1.csv",
    "archive/FilteredEvals2.csv",
    "archive/FilteredEvals3.csv"
]

kaggle = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
kaggle = kaggle.rename(columns={"FEN":"fen","Evaluation":"evaluation"})
print(f"Kaggle rows : {len(kaggle)}")

kaggle["key"] = fen_key(kaggle["fen"])
kaggle = kaggle.drop_duplicates("key").reset_index(drop=True)
print(f"Kaggle rows after removing duplicates : {len(kaggle)}")

Kaggle rows : 7920728
Kaggle rows after removing duplicates : 7675371


In [4]:
own = pd.read_csv("dataset.csv")[["fen","evaluation"]]
print(f"Own rows : {len(own)}")

own["key"] = fen_key(own["fen"])
own = own.drop_duplicates("key")
print(f"Own rows after removing duplicates : {len(own)}")

own = own[~own["key"].isin(set(kaggle["key"]))].reset_index(drop=True)
print(f"Own rows after removing positions also in Kaggle : {len(own)}")

Own rows : 100000
Own rows after removing duplicates : 100000
Own rows after removing positions also in Kaggle : 96771


In [5]:
k_train, k_val, k_test = split(kaggle)
o_train, o_val, o_test = split(own)

for name, df in [("kaggle_train",k_train),("kaggle_val",k_val),("kaggle_test",k_test),
                 ("own_train",o_train),("own_val",o_val),("own_test",o_test)]:
    df[["fen","evaluation"]].to_csv(out_dir / f"{name}.csv", index=False)
    print(f"{name:13s} : {len(df)}")

kaggle_train  : 6140297
kaggle_val    : 767537
kaggle_test   : 767537
own_train     : 77417
own_val       : 9677
own_test      : 9677


In [6]:
y = np.clip(k_test["evaluation"].values, -1000, 1000)
print(f"Kaggle test |eval| > 1000 before clipping : {np.mean(np.abs(k_test['evaluation'].values) > 1000):.4f}")
print(f"Kaggle test eval std (clipped) : {y.std():.2f}")

Kaggle test |eval| > 1000 before clipping : 0.0366
Kaggle test eval std (clipped) : 401.64
